# Part 2 - Data Visualization

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
os.makedirs("mp2_outputs", exist_ok=True)


In [2]:
df = pd.read_csv("cdamron3_project_summary.csv", sep=";")
print("\n")
print(df.shape)
print("\n")
print("Columns:\n", df.columns.tolist())
print("\n")
print(df.head())



(25776, 5)


Columns:
 ['project_wocid', 'commit_sha1', 'author', 'time', 'commit_message']


           project_wocid                               commit_sha1  \
0  marianattestad_ribbon  0023779e992f3962eecf62e652ba81fe66da3852   
1  marianattestad_ribbon  00c7ec6d1edf1ee4fef4e87b9e8254271f76fd2e   
2  marianattestad_ribbon  013cfce4dde223bddcc8a599db060b3ee06a8881   
3  marianattestad_ribbon  01ded9b5bd9afe66e5469d7864b8fbe83bb1e66c   
4  marianattestad_ribbon  03ea1c89e7669fd6942131c747023ffa08e7372b   

                                            author        time  \
0  Robert Aboukhalil <robert.aboukhalil@gmail.com>  1584246680   
1      Maria Nattestad <maria.nattestad@gmail.com>  1472156867   
2  Robert Aboukhalil <robert.aboukhalil@gmail.com>  1725660419   
3      Maria Nattestad <maria.nattestad@gmail.com>  1472576123   
4  Robert Aboukhalil <robert.aboukhalil@gmail.com>  1723753639   

                                      commit_message  
0                              

In [3]:
df = df.rename(columns= {
    "project_wocid": "project",
    "commit_sha1": "commit",
    "message": "message",
    "commit_message": "message",
})

# df.columns = ['project','commit','author','time','message']

df = df[["project", "commit", "author", "time", "message"]]
df['Month'] = pd.to_datetime(df['time'], unit='s').dt.strftime('%Y-%m')
df.head()

,project,commit,author,time,message,Month
0,marianattestad_ribbon,0023779e992f3962eecf62e652ba81fe66da3852,Robert Aboukhalil <robert.aboukhalil@gmail.com>,1584246680,Small fix\n,2020-03
1,marianattestad_ribbon,00c7ec6d1edf1ee4fef4e87b9e8254271f76fd2e,Maria Nattestad <maria.nattestad@gmail.com>,1472156867,Merge branch 'master' into web-production\n,2016-08
2,marianattestad_ribbon,013cfce4dde223bddcc8a599db060b3ee06a8881,Robert Aboukhalil <robert.aboukhalil@gmail.com>,1725660419,Download & mount BAIs in one shot instead of 1...,2024-09
3,marianattestad_ribbon,01ded9b5bd9afe66e5469d7864b8fbe83bb1e66c,Maria Nattestad <maria.nattestad@gmail.com>,1472576123,Add variants and features in a singleview head...,2016-08
4,marianattestad_ribbon,03ea1c89e7669fd6942131c747023ffa08e7372b,Robert Aboukhalil <robert.aboukhalil@gmail.com>,1723753639,Fix bug with saveSvgAsPng in splitthreader\n,2024-08


In [4]:
projects = sorted(df["project"].unique())
print(f"\Found {len(projects)} projeccts: {projects}\n")

gap_count = {}

for prj in projects:
    print(f"Loop Start - Current Project {prj}\n")
    df1 = df[df["project"] == prj]

    df2 = df1.groupby("Month").size().reset_index(name="Commits")

    df2["time"] = pd.to_datetime(df2["Month"])
    full_range = pd.DataFrame({"time": pd. date_range(start = df2["time"].min(), end=df2["time"].max(), freq="MS")})

    mc = pd.merge(full_range, df2, on="time", how="left")
    mc["Commits"] = mc["Commits"].fillna(0).astype(int)
    mc["Month"] = mc["time"].dt.strftime("%Y-%m")
    mc = mc[["Month", "Commits"]]

    ts_path = f"mp2_outputs/cdamron3_commits_timeseries_{prj}.csv"
    mc.to_csv(ts_path, sep=";", index=False)

    plt.figure(figsize=(12, 5))
    plt.plot(mc["Month"], mc["Commits"],
             marker="o", linestyle="-", markersize=3)
    plt.xlabel("Time (YYYY-MM)")
    plt.ylabel("Number of Commits")
    plt.title(f"Commit Activity: {prj}")
    plt.grid(True)
    plt.tight_layout()
    ax = plt.gca()

    for i, label in enumerate(ax.xaxis.get_ticklabels()):
       if i % 2 == 1:
            label.set_visible(False)
    plt.xticks(rotation=45, fontsize=7)
    plt.savefig(f"mp2_outputs/cdamron3_timeseries_{prj}.png", dpi=600)
    plt.close()

    commits = mc["Commits"].tolist()
    months = mc["Month"].tolist()

    max_len = 0
    curr_len = 0
    curr_start = None
    best_start = None
    best_end = None

    for m, c in zip(months, commits):
        if c == 0:
            if curr_len == 0:
                curr_start = m
            curr_len += 1
            if curr_len > max_len:
                max_len = curr_len
                best_start = curr_start
                best_end = m
        else:
            curr_len = 0

    num_gaps = 0
    curr = 0
    for c in commits:
        if c == 0:
            curr += 1
        else:
            if curr >= 3:
               num_gaps += 1
            curr = 0
    if curr >= 3:
        num_gaps += 1

    commits_after = 0
    if best_end:
       index = months.index(best_end)
       commits_after = sum(commits[index+1:])

    gap_count[prj] = {
        "LongestGapStart": best_start,
        "LongestGapEnd": best_end,
        "LongestGapLength": max_len,
        "NumCommitsAfterLastGap": commits_after,
        "NumGapsTimeline": num_gaps,
    }

    print(f" Longest Gap: {best_start} -> {best_end} ({max_len} months)\n")
    print(f" gaps >= 3 months: {num_gaps}, commits after gap: {commits_after}\n")

gap_df = pd.DataFrame.from_dict(gap_count, orient="index").reset_index()
gap_df.rename(columns={"index": "Project"}, inplace=True)
gap_df.to_csv(f"mp2_outputs/cdamron3_gap_summary.csv", sep=";", index=False)

print("Done!")

\Found 10 projeccts: ['jhuapl-boss_intern', 'jonescompneurolab_hnn', 'lpwgroup_crank', 'marianattestad_ribbon', 'mirnylab_cooler', 'privefl_bigstatsr', 'rveltz_piecewisedeterministicmarkovprocesses.jl', 'stan-dev_cmdstanpy', 'thelfer_mfrontgenericinterfacesupport', 'xeokit_xeokit-sdk']

Loop Start - Current Project jhuapl-boss_intern

 Longest Gap: 2024-01 -> 2024-05 (5 months)

 gaps >= 3 months: 6, commits after gap: 20

Loop Start - Current Project jonescompneurolab_hnn

 Longest Gap: 2022-09 -> 2023-09 (13 months)

 gaps >= 3 months: 3, commits after gap: 73

Loop Start - Current Project lpwgroup_crank

 Longest Gap: 2022-04 -> 2023-03 (12 months)

 gaps >= 3 months: 6, commits after gap: 20

Loop Start - Current Project marianattestad_ribbon

 Longest Gap: 2017-11 -> 2019-09 (23 months)

 gaps >= 3 months: 7, commits after gap: 302

Loop Start - Current Project mirnylab_cooler

 Longest Gap: 2021-05 -> 2021-10 (6 months)

 gaps >= 3 months: 5, commits after gap: 401

Loop Start - 

In [5]:
# Just to Check Everything:

files = sorted(os.listdir("mp2_outputs"))
print(f"{len(files)} files in mp2_outputs:\n")
for f in files:
    print(" ", f)


22 files in mp2_outputs:

  .ipynb_checkpoints
  cdamron3_commits_timeseries_jhuapl-boss_intern.csv
  cdamron3_commits_timeseries_jonescompneurolab_hnn.csv
  cdamron3_commits_timeseries_lpwgroup_crank.csv
  cdamron3_commits_timeseries_marianattestad_ribbon.csv
  cdamron3_commits_timeseries_mirnylab_cooler.csv
  cdamron3_commits_timeseries_privefl_bigstatsr.csv
  cdamron3_commits_timeseries_rveltz_piecewisedeterministicmarkovprocesses.jl.csv
  cdamron3_commits_timeseries_stan-dev_cmdstanpy.csv
  cdamron3_commits_timeseries_thelfer_mfrontgenericinterfacesupport.csv
  cdamron3_commits_timeseries_xeokit_xeokit-sdk.csv
  cdamron3_gap_summary.csv
  cdamron3_timeseries_jhuapl-boss_intern.png
  cdamron3_timeseries_jonescompneurolab_hnn.png
  cdamron3_timeseries_lpwgroup_crank.png
  cdamron3_timeseries_marianattestad_ribbon.png
  cdamron3_timeseries_mirnylab_cooler.png
  cdamron3_timeseries_privefl_bigstatsr.png
  cdamron3_timeseries_rveltz_piecewisedeterministicmarkovprocesses.jl.png
  cdamron

In [6]:
with open("mp2_outputs/cdamron3_commits_timeseries_marianattestad_ribbon.csv") as f:
    for _ in range(5):
        print(f.readline().rstrip())


Month;Commits
2016-06;4
2016-07;35
2016-08;100
2016-09;34


### Interpretation:

| Project | Longest gap | # gaps ≥ 3 mo | Pattern |
|---|---|---|---|
| jhuapl-boss_intern | 2024-01 → 2024-05 (5 mo) | 6 | declining |
| jonescompneurolab_hnn | 2022-09 → 2023-09 (13 mo) | 3 | irregular |
| lpwgroup_crank | 2022-04 → 2023-03 (12 mo) | 6 | irregular |
| marianattestad_ribbon | 2017-11 → 2019-09 (23 mo) | 7 | U-shaped |
| mirnylab_cooler | 2021-05 → 2021-10 (6 mo) | 5 | declining |
| privefl_bigstatsr | 2022-11 → 2023-11 (13 mo) | 4 | declining |
| rveltz_piecewisedeterministicmarkovprocesses.jl | 2022-11 → 2023-12 (14 mo) | 10 | irregular |
| stan-dev_cmdstanpy | 2 (which is not long enough) | 0 | irregular |
| thelfer_mfrontgenericinterfacesupport | 2022-07 → 2022-11 (5 mo) | 2 | U-shaped |
| xeokit_xeokit-sdk | none | 0 | irregular |

### Notes for each Project:

- **jhuapl-boss_intern** — Somewhat annual activity with decline overall. The project starts with a huge burst in activity that goes around 190ish commits before quickly sliding back down at the end of 2016. These spikes continue to appear for 4 months after a year of very low activity, which one could say is irregular, but the general trend is a slow decrease in the number of commits in both the peak of spikes and the years of almost nothing.
  
- **jonescompneurolab_hnn** Generally high activity from 2017 to 2021, then very low activity from around 2021 onwards. From 2017 to 2021, the project goes from months of massive peaks to months of low activity, but generally overall stays relatively high in terms of commits. After 2021, the general overall stays mostly low with the highest number of commits in the second half being 30 compared to the highest of the first half around 125.
  
- **lpwgroup_crank** — Starts with high peaks for around the first year before remaining extremely low after 2019 outside of random increases. 
  
- **marianattestad_ribbon** — Almost has a U or W like shape as the two highest peaks are at the beginning and end, with a small surge in activity around the middle.
  
- **mirnylab_cooler** — Within the first six months of the porject, the highest amount of commits is achieved at around 115 and afterwards is a somewhat steady decline outside of one outlier 3/4ths of the way into the activity graph.
  
- **privefl_bigstatsr** — Within around the first year of activity between 2016-2017, the highest number of commits is hit at around 95 and slowly declines after with some variation in the peaks and valleys, but overall still a steady decrease.
  
- **rveltz_piecewisedeterministicmarkovprocesses.jl** — Similar to jonescompneurolab_hnn, the amount of commits remains relatively constantly high between 2015 and 2019, but after 2019, the number of commits falls off a cliff.
  
- **stan-dev_cmdstanpy** — While one could argue that this project has a general trend of decline, the majority of the graph has peaks and valleys that don't correlate to much outside of there being consistently higher amounts of commits from 2019 to 2022, but then a dropoff after 2022.
  
- **thelfer_mfrontgenericinterfacesupport** — Within the first third of the project's activity, it hits a peak of around 67 commits and then hits a similar peak at the very end of the activity graph. All other months remain in low actiivty, ranging from around 25 to 0 commits.
  
- **xeokit_xeokit-sdk** — There is little to no activity from 2018 to 2024, before a huge surge in commits going all the way up to around 2500 in the last 4th of the activity graph and remains close to that until the end.

# Part 3.1 - Examining Commits between Gaps

In [2]:
import pandas as pd

try:
    df
except NameError:
    df = pd.read_csv(f"cdamron3_project_summary.csv", sep=";")
    df = df.rename(columns={
        "project_wocid":  "project",
        "commit_sha1":    "commit",
        "commit_message": "message",
    })
    df = df[["project", "commit", "author", "time", "message"]]
    df["time"] = pd.to_datetime(df["time"], unit="s")

gap_df = pd.read_csv(f"mp2_outputs/cdamron3_gap_summary.csv", sep=";")

rows = []
for _, r in gap_df.iterrows():
    prj       = r["Project"]
    gap_start = r["LongestGapStart"]
    gap_end   = r["LongestGapEnd"]

    if pd.isna(gap_start) or pd.isna(gap_end):
        print(f"{prj}: no qualifying gap -> skipped")
        continue

    proj = df[df["project"] == prj].sort_values("time")
    start_cut = pd.to_datetime(gap_start)
    end_cut = pd.to_datetime(gap_end) + pd.offsets.MonthEnd(0)

    pre  = proj[proj["time"] <  start_cut].tail(10).copy()
    post = proj[proj["time"] >  end_cut].head(10).copy()

    pre["pre/post"]  = "pre"
    post["pre/post"] = "post"

    rows.append(pre)
    rows.append(post)
    print(f"{prj}: {len(pre)} pre, {len(post)} post")

gap_commits = pd.concat(rows, ignore_index=True)
gap_commits = gap_commits[["pre/post", "project", "commit", "author", "time", "message"]]

out = f"mp2_outputs/cdamron3_project_gap_commits.csv"
gap_commits.to_csv(out, sep=";", index=False)
print(f"\nWrote {len(gap_commits)} rows -> {out}")


jhuapl-boss_intern: 10 pre, 10 post
jonescompneurolab_hnn: 10 pre, 10 post
lpwgroup_crank: 10 pre, 10 post
marianattestad_ribbon: 10 pre, 10 post
mirnylab_cooler: 10 pre, 10 post
privefl_bigstatsr: 10 pre, 10 post
rveltz_piecewisedeterministicmarkovprocesses.jl: 10 pre, 10 post
stan-dev_cmdstanpy: 10 pre, 10 post
thelfer_mfrontgenericinterfacesupport: 10 pre, 10 post
xeokit_xeokit-sdk: no qualifying gap -> skipped

Wrote 180 rows -> mp2_outputs/cdamron3_project_gap_commits.csv


# Part 3.2 - Interpret the Gap

In [3]:
import pandas as pd

# ---------- Part 1: basic stats from your summary CSV ----------
df = pd.read_csv(f"cdamron3_project_summary.csv", sep=";")
df = df.rename(columns={
    "project_wocid":  "project",
    "commit_sha1":    "commit",
    "commit_message": "message",
})
df["time"] = pd.to_datetime(df["time"], unit="s")

part1 = df.groupby("project").agg(
    ncommits=("commit", "count"),
    nauthors=("author", "nunique"),
    from_time=("time", "min"),
    to_time=("time", "max"),
).reset_index()
part1 = part1.rename(columns={
    "project":   "Project",
    "from_time": "from",
    "to_time":   "to",
})

# ---------- Part 2: gap summary ----------
gap_df = pd.read_csv(f"mp2_outputs/cdamron3_gap_summary.csv", sep=";")

# ---------- Merge ----------
stats = pd.merge(part1, gap_df, on="Project", how="left")

# ---------- Placeholders for the manual columns you'll fill in later ----------
manual_cols = [
    "nstars", "nforks", "lastGHCommitDate",       # Part 1 (GitHub lookups)
    "ActivityPattern",                             # Part 2 Interpretation
    "BeforeThemes", "AfterThemes",                 # Part 3.1
    "HypothesizedGapReason", "HasRecovered",
    "WhyRecovered", "WhoRecovered",                # Part 3.2
    "Currentstatus", "RecentThemes", "Notes",      # Part 4
]
for c in manual_cols:
    if c not in stats.columns:
        stats[c] = ""

# ---------- Reorder to match the spec ----------
final_cols = [
    "Project", "ncommits", "nauthors", "from", "to",
    "nstars", "nforks", "lastGHCommitDate",
    "LongestGapStart", "LongestGapEnd", "LongestGapLength",
    "NumCommitsAfterLastGap", "ActivityPattern", "NumberOfGapsInTimeline",
    "BeforeThemes", "AfterThemes",
    "HypothesizedGapReason", "HasRecovered", "WhyRecovered", "WhoRecovered",
    "Currentstatus", "RecentThemes", "Notes",
]
stats = stats.reindex(columns=final_cols)

# ---------- Save ----------
out = f"mp2_outputs/cdamron3_project_stats.csv"
stats.to_csv(out, sep=";", index=False)

print(f"Created {out} with {len(stats)} rows and {len(stats.columns)} columns")
print("\nColumns:")
print(stats.columns.tolist())
print("\nFirst few rows (basic columns):")
print(stats[["Project", "ncommits", "nauthors",
             "LongestGapStart", "LongestGapEnd", "LongestGapLength"]])


Created mp2_outputs/cdamron3_project_stats.csv with 10 rows and 23 columns

Columns:
['Project', 'ncommits', 'nauthors', 'from', 'to', 'nstars', 'nforks', 'lastGHCommitDate', 'LongestGapStart', 'LongestGapEnd', 'LongestGapLength', 'NumCommitsAfterLastGap', 'ActivityPattern', 'NumberOfGapsInTimeline', 'BeforeThemes', 'AfterThemes', 'HypothesizedGapReason', 'HasRecovered', 'WhyRecovered', 'WhoRecovered', 'Currentstatus', 'RecentThemes', 'Notes']

First few rows (basic columns):
                                           Project  ncommits  nauthors  \
0                               jhuapl-boss_intern      1471        45   
1                            jonescompneurolab_hnn      2049        39   
2                                   lpwgroup_crank       370        27   
3                            marianattestad_ribbon       519         7   
4                                  mirnylab_cooler      1403        47   
5                                privefl_bigstatsr      1007        13   
6

In [19]:
import pandas as pd
stats = pd.read_csv("mp2_outputs/cdamron3_project_stats.csv", sep=";")
stats

,Project,ncommits,nauthors,from,to,nstars,nforks,lastGHCommitDate,LongestGapStart,LongestGapEnd,...,NumberOfGapsInTimeline,BeforeThemes,AfterThemes,HypothesizedGapReason,HasRecovered,WhyRecovered,WhoRecovered,Currentstatus,RecentThemes,Notes
0,jhuapl-boss_intern,1471,45,2015-09-30 14:14:23,2025-04-28 16:55:42,23,15,2025-04-18,2024-01,2024-05,...,6,Bug fixes:Feature development,Bug fixes:Feature development,Post-release wind-down: after tagging v1.4.1 (...,Yes,New bug reports / feature requests arrived; Jo...,"Same maintainers (Jordan Matelsky, Daniel Xenes)",Active,Bug fixes:Release,Activity is within bursts and declining relati...
1,jonescompneurolab_hnn,2049,39,2017-02-23 15:44:55,2025-10-28 21:45:29,94,26,2025-10-28,2022-09,2023-09,...,3,Documentation updates:Feature development,Other:Documentation updates,Maintainer transition: pre-gap commits in Aug ...,Yes,New contributor Josh Lu resumed work in Oct 20...,New contributors (Josh Lu / jashlu) driving bu...,Active,Documentation updates:Bug fixes,The project shows a clear two-phase pattern: h...
2,lpwgroup_crank,370,27,2018-03-04 06:08:28,2025-05-03 06:47:18,38,22,2025-05-03,2022-04,2023-03,...,6,Bug fixes:Feature development,Feature development:Bug fixes,Contributor-driven: bulk of pre-gap work came ...,Yes,Zhiyi Wu returned in Apr 2023 to add xTB suppo...,Same key contributor (Zhiyi Wu) plus new contr...,Active,Release:Bug fixes,Activity is irregular with 6 distinct gaps and...
3,marianattestad_ribbon,519,7,2016-06-30 00:31:08,2025-06-28 08:30:38,306,32,2026-08-19,2017-11,2019-09,...,7,Bug fixes:Feature development,Dependency updates:Feature development,Maintainer career change: Maria Nattestad was ...,Yes,Robert Aboukhalil revived the project in Oct 2...,New primary contributor (Robert Aboukhalil); o...,Active,Feature development:Documentation updates,"This project has a strong U-shaped pattern, wh..."
4,mirnylab_cooler,1403,47,2015-11-16 19:54:19,2025-11-03 16:28:29,245,62,2026-01-28,2021-05,2021-10,...,5,Bug fixes:Dependency updates,Feature development:Dependency updates,Post-release pause: pre-gap commits in April 2...,Yes,Nezar Abdennur returned in Nov 2021 with major...,Same maintainers (Nezar Abdennur),Active,Dependency updates:Other,Activity is heavily front-loaded from 2015 to ...
5,privefl_bigstatsr,1007,13,2016-09-15 06:50:51,2025-07-29 09:30:36,180,30,2026-06-05,2022-11,2023-11,...,4,Bug fixes:Dependency updates,Bug fixes:Dependency updates,Post-CRAN-release pause: last pre-gap commit (...,Yes,User-reported issues (Andrea Manica's variance...,Same maintainer (Florian Prive) responding to ...,Active,Bug fixes:Dependency updates,The project has a sort of steady/declining pat...
6,rveltz_piecewisedeterministicmarkovprocesses.jl,608,14,2015-12-16 12:56:55,2026-03-06 07:54:59,21,8,2026-06-05,2022-11,2023-12,...,10,Dependency updates:Release,Dependency updates:Bug fixes,Solo maintainer's periodic maintenance cycle: ...,Yes,Julia ecosystem drift (DiffEqBase to SciMLBase...,Same maintainer (romain veltz),Active,Dependency updates:Bug fixes,This project is a soley maintained one that ha...
7,stan-dev_cmdstanpy,2742,83,2019-01-03 00:01:50,2025-11-05 17:34:16,109,81,2026-10-06,2024-07,2024-08,...,0,Documentation updates:Bug fixes,Feature development:Bug fixes,N/A active Project (longest run of zeros was 2...,NaN,NaN,NaN,Active,Bug fixes:Feature development,No qualifying inactivity gap (longest run of z...
8,thelfer_mfrontgenericinterfacesupport,751,25,2018-06-20 12:42:33,2025-11-05 21:28:11,41,36,2026-10-02,2022-07,2022-11,...,2,Bug fixes:Other,Bug fixes:Feature development,Post-release pause: pre-gap commits (2022-06-1...,Yes,New feature requests around material propertie...,Same maintainer (Thomas Helfer) plus external ...,Active,Bug fixes:Feature development,A U-shaped pattern seems to fit this project a...
9,xeokit_xeokit-sdk,14856,116,2018-12-01 17:26:31,2025-11-05 21:41:25,944,333,2026-10-05,NaN,NaN,...,0,N/A active Project,N/A active Project,N/A active Project (no 

In [21]:
stats = pd.read_csv("mp2_outputs/cdamron3_project_stats.csv", sep=";")

manual_cols = [
    "nstars", "nforks", "lastGHCommitDate",
    "ActivityPattern", "BeforeThemes", "AfterThemes",
    "HypothesizedGapReason", "HasRecovered", "WhyRecovered", "WhoRecovered",
    "Currentstatus", "RecentThemes", "Notes",
]
for col in manual_cols:
    if col in stats.columns:
        stats[col] = stats[col].astype("object")

fills = {
    "jhuapl-boss_intern": {
        "nstars": 23, "nforks": 15, "lastGHCommitDate": "2025-04-18",
        "ActivityPattern": "declining",
        "BeforeThemes": "Bug fixes:Feature development",
        "AfterThemes": "Bug fixes:Feature development",
        "HypothesizedGapReason": "Post-release wind-down: after tagging v1.4.1 (2023-03) and a final round of small fixes and CI updates in late 2023, the immediate work queue was empty. Maintainers stepped back while waiting for new demand.",
        "HasRecovered": "Yes",
        "WhyRecovered": "New bug reports / feature requests arrived; Jordan Matelsky returned in June 2024 to add a CloudVolume-backed link fix and to update CI/Python version support in early 2025.",
        "WhoRecovered": "Same maintainers (Jordan Matelsky, Daniel Xenes)",
        "Currentstatus": "Active",
        "RecentThemes": "Bug fixes:Release",
        "Notes": "Project had a 5-month gap (2024-01 to 2024-05) after a release cycle ended. Recovered quickly with the same two maintainers; activity has continued at a low but steady rate.",
    },
    "jonescompneurolab_hnn": {
        "nstars": 94, "nforks": 26, "lastGHCommitDate": "2025-10-28",
        "ActivityPattern": "irregular",
        "BeforeThemes": "Documentation updates:Feature development",
        "AfterThemes": "Other:Documentation updates",
        "HypothesizedGapReason": "Maintainer transition: pre-gap commits in Aug 2022 were doc-only edits by rythorpe and a WIP by Mainak Jas (hnn-core compatibility refactor). A 13-month gap follows, suggesting a pause while larger refactoring work happened off-main or the primary maintainer moved on.",
        "HasRecovered": "Yes",
        "WhyRecovered": "New contributor Josh Lu resumed work in Oct 2023 with build/CI updates (dockerfile, xquartz), signaling the next generation of lab members picked it back up.",
        "WhoRecovered": "New contributors (Josh Lu / jashlu) driving build and docker maintenance",
        "Currentstatus": "Active",
        "RecentThemes": "Documentation updates:Bug fixes",
        "Notes": "Longest gap (13 months) coincides with a maintainer handoff; recovery was driven by new lab contributors doing build tooling rather than original feature authors. Project remains lightly maintained.",
    },
    "lpwgroup_crank": {
        "nstars": 38, "nforks": 22, "lastGHCommitDate": "2025-05-03",
        "ActivityPattern": "irregular",
        "BeforeThemes": "Bug fixes:Feature development",
        "AfterThemes": "Feature development:Bug fixes",
        "HypothesizedGapReason": "Contributor-driven: bulk of pre-gap work came from Zhiyi Wu's Gaussian/torsiondrive bug fixes in Jan-Mar 2022. Once Zhiyi's contributions paused, the project went quiet until new features were requested.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Zhiyi Wu returned in Apr 2023 to add xTB support, and new contributor Jessica Flowers added an ASE engine in mid-2023, driven by external feature requests.",
        "WhoRecovered": "Same key contributor (Zhiyi Wu) plus new contributor (Jessica Flowers)",
        "Currentstatus": "Active",
        "RecentThemes": "Release:Bug fixes",
        "Notes": "Gap (2022-04 to 2023-03) aligns with a pause in the primary external contributor's activity. Recovery came from a new research need (xTB + ASE engines).",
    },
    "marianattestad_ribbon": {
        "nstars": 306, "nforks": 32, "lastGHCommitDate": "2026-08-19",
        "ActivityPattern": "U-shaped",
        "BeforeThemes": "Bug fixes:Feature development",
        "AfterThemes": "Dependency updates:Feature development",
        "HypothesizedGapReason": "Maintainer career change: Maria Nattestad was the sole active author through 2017. A 23-month silence follows, likely because the author moved to a new role and the project went dormant.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Robert Aboukhalil revived the project in Oct 2019, migrating it to WebAssembly (samtools.wasm, Aioli) and modern npm packaging.",
        "WhoRecovered": "New primary contributor (Robert Aboukhalil); original author Maria Nattestad returned later for documentation/release work",
        "Currentstatus": "Active",
        "RecentThemes": "Feature development:Documentation updates",
        "Notes": "Classic 23-month abandonment-then-revival: original maintainer stepped away in 2017, new contributor modernized the stack in 2019, original author eventually rejoined. The gap was relatively easy to interpret from commit messages.",
    },
    "mirnylab_cooler": {
        "nstars": 245, "nforks": 62, "lastGHCommitDate": "2026-01-28",
        "ActivityPattern": "declining",
        "BeforeThemes": "Bug fixes:Dependency updates",
        "AfterThemes": "Feature development:Dependency updates",
        "HypothesizedGapReason": "Post-release pause: pre-gap commits in April 2021 were a version bump, changelog update, and CI addition - end-of-release-cycle cleanup. Maintainers stepped back while no urgent work was queued.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Nezar Abdennur returned in Nov 2021 with major internal refactoring (new query engine, deprecation compatibility) planned during the quiet period.",
        "WhoRecovered": "Same maintainers (Nezar Abdennur)",
        "Currentstatus": "Active",
        "RecentThemes": "Dependency updates:Other",
        "Notes": "6-month gap (2021-05 to 2021-10) was a release-cycle lull. Recovery driven by the same maintainer with an internal engine rewrite. Recent activity is largely automated (dependabot, pre-commit-ci).",
    },
    "privefl_bigstatsr": {
        "nstars": 180, "nforks": 30, "lastGHCommitDate": "2026-06-05",
        "ActivityPattern": "declining",
        "BeforeThemes": "Bug fixes:Dependency updates",
        "AfterThemes": "Bug fixes:Dependency updates",
        "HypothesizedGapReason": "Post-CRAN-release pause: last pre-gap commit (2022-10-14) was 'fixes for CRAN'. A single-maintainer project goes quiet after shipping, waiting for downstream bug reports.",
        "HasRecovered": "Yes",
        "WhyRecovered": "User-reported issues (Andrea Manica's variance-zero bug, issue #170 for backingfile) triggered fixes in Dec 2023 and Mar 2024.",
        "WhoRecovered": "Same maintainer (Florian Prive) responding to user reports; occasional external contributors (Andrea Manica)",
        "Currentstatus": "Active",
        "RecentThemes": "Bug fixes:Dependency updates",
        "Notes": "13-month gap (2022-11 to 2023-11) followed a CRAN release. Recovery was reactive, driven by incoming bug reports. Project rhythm is 'release, quiet, fix, release'.",
    },
    "rveltz_piecewisedeterministicmarkovprocesses.jl": {
        "nstars": 21, "nforks": 8, "lastGHCommitDate": "2026-06-05",
        "ActivityPattern": "irregular",
        "BeforeThemes": "Dependency updates:Release",
        "AfterThemes": "Dependency updates:Bug fixes",
        "HypothesizedGapReason": "Solo maintainer's periodic maintenance cycle: pre-gap burst in Oct 2022 ended with a v0.0.7 release. Romain Veltz, the sole contributor, went quiet for 14 months - typical of a Julia package updated only when dependency compatibility breaks.",
        "HasRecovered": "Yes",
        "WhyRecovered": "Julia ecosystem drift (DiffEqBase to SciMLBase, JumpProcesses) required dependency updates starting Jan 2024.",
        "WhoRecovered": "Same maintainer (romain veltz)",
        "Currentstatus": "Active",
        "RecentThemes": "Dependency updates:Bug fixes",
        "Notes": "10 gaps >= 3 months total - this is a solo-maintained academic package that only updates when Julia dependencies require it. The 14-month gap is typical, not anomalous.",
    },
    "stan-dev_cmdstanpy": {
        "nstars": 109, "nforks": 81, "lastGHCommitDate": "2026-10-06",
        "ActivityPattern": "irregular",
        "BeforeThemes": "Documentation updates:Bug fixes",
        "AfterThemes": "Feature development:Bug fixes",
        "HypothesizedGapReason": "N/A active Project (longest run of zeros was 2 months, below the 3-month gap threshold)",
        "HasRecovered": "N/A (no inactivity gap existed)",
        "WhyRecovered": "N/A (no inactivity gap existed)",
        "WhoRecovered": "N/A (no inactivity gap existed)",
        "Currentstatus": "Active",
        "RecentThemes": "Bug fixes:Feature development",
        "Notes": "Continuous development since 2019 with no qualifying inactivity gap. Multiple active maintainers (Brian Ward, Mitzi Morris, Ari Hartikainen) plus external contributors.",
    },
    "thelfer_mfrontgenericinterfacesupport": {
        "nstars": 41, "nforks": 36, "lastGHCommitDate": "2026-10-02",
        "ActivityPattern": "U-shaped",
        "BeforeThemes": "Bug fixes:Other",
        "AfterThemes": "Bug fixes:Feature development",
        "HypothesizedGapReason": "Post-release pause: pre-gap commits (2022-06-17) were formatting, cleanups, and a PR merge. A 5-month silence follows.",
        "HasRecovered": "Yes",
        "WhyRecovered": "New feature requests around material properties (Issue #106, PR #107) pulled the maintainer back in Dec 2022, alongside external collaborators.",
        "WhoRecovered": "Same maintainer (Thomas Helfer) plus external collaborators (Lars Bilke, Mathieu Courtois)",
        "Currentstatus": "Active",
        "RecentThemes": "Bug fixes:Feature development",
        "Notes": "Short 5-month gap ended with a planned new feature. External collaborators (OGS, EDF) contribute to specific features, keeping the project active.",
    },
    "xeokit_xeokit-sdk": {
        "nstars": 944, "nforks": 333, "lastGHCommitDate": "2026-10-05",
        "ActivityPattern": "irregular",
        "BeforeThemes": "N/A active Project",
        "AfterThemes": "N/A active Project",
        "HypothesizedGapReason": "N/A active Project (no inactivity gap existed)",
        "HasRecovered": "N/A (no inactivity gap existed)",
        "WhyRecovered": "N/A (no inactivity gap existed)",
        "WhoRecovered": "N/A (no inactivity gap existed)",
        "Currentstatus": "Active",
        "RecentThemes": "Feature development:Documentation updates",
        "Notes": "Continuously active since 2018 with zero gaps. Commercial backing (Creoox) and a large contributor base (116 authors) sustain development.",
    },
}

for prj, cols in fills.items():
    for col, val in cols.items():
        stats.loc[stats["Project"] == prj, col] = val

stats.to_csv("mp2_outputs/cdamron3_project_stats.csv", sep=";", index=False)
print("Done. Filled", len(fills), "projects.")


Done. Filled 10 projects.


In [18]:
stats = pd.read_csv("mp2_outputs/cdamron3_project_stats.csv", sep=";")

notes = {
    "jhuapl-boss_intern":
        "Activity is within bursts and declining relative to its early peak as numerous amounts of of feature and bug-fix commits are separated by long quiet stretches. The 5-month gap (2024-01 to 2024-05) was easy to interpret as the last pre-gap commits were a release tag followed by small fixes and CI cleanups. This makes sense as a natural wind-down after finishing a work cycle, and the very first post-gap commit shows the same two maintainers returning when a new bug surfaced. Recovery was fast and did not involve new contributors, suggesting the project is simply low-traffic between requests rather than abandoned.",

    "jonescompneurolab_hnn":
        "The project shows a clear two-phase pattern: high sustained activity (2017 to 2021) followed by a sharp drop-off and a 13-month silence. This gap was quite hard to interpret from commit messages alone as the last pre-gap commits were documentation edits and something about a WIP, which hint at a larger codebase overhaul happening off the main branch rather than total abandonment. Recovery was driven by new lab members doing build/docker tooling rather than the original feature authors, suggesting the maintainer(s) handed off the work within the research group rather than a project revival by outsiders.",

    "lpwgroup_crank":
        "Activity is irregular with 6 distinct gaps and no consistent cadence. The longest gap was moderately easy to interpret as nearly all pre-gap commits were concentrated in a short January-March 2022 window by external contributor Zhiyi Wu, and once that contributor paused, the repo went silent. This implies the project's activity was driven by a single outside contributor rather than the main maintainer. Recovery came when Zhiyi Wu returned to add support for a feature, and a new contributor named Jessica Flowers added an engine soon after.",

    "marianattestad_ribbon":
        "This project has a strong U-shaped pattern, which is odd. It seems heavy development happend in 2016 and then a 23-month silence began, with a strong revival from 2019 to 2026. The gap was easy to interpret as Maria Nattestad was the sole active author pre-gap and no other contributors were thre to take her place, making the project dormant. Recovery occured when Robert Aboukhalil picked up the codebase in Oct 2019 and modernized it. The original author eventually returned for documentation and release work it seems.",

    "mirnylab_cooler":
        "Activity is heavily front-loaded from 2015 to 2018 and declined significantly afterward. The longest gap falling in the middle of the slowdown. The gap was easy to interpret as pre-gap commits were a version bump, changelog update, and a CI addition which suggests the end of a release. The first post-gap commits were a large internal refactor that had likely been under development during the quiet period. Recovery was by the same maintainers, so it seems like the absence was planned before the larger changes rather than abandonment. Notably, recent activity is dominated by bots which keeps the project 'active' on GitHub even without much manual development.",

    "privefl_bigstatsr":
        "The project has a sort of steady/declining pattern. Work comes in bursts and is separated by long quiet periods. The 13-month gap was easy to interpret as the last pre-gap commit was literally the terminal step of a release. Recovery began soon after when external users filed bug reports, triggering reactive fixes. The maintainer is suprisingly still responsive but not proactively developing new features.",

    "rveltz_piecewisedeterministicmarkovprocesses.jl":
        "This project is a soley maintained one that has quite the irregular pattern as there are tons of gaps. The gap I looked at was easy to interpret from the commit messages alone as the pre-gap burst ended with a 'new version 0.0.7' release, and the post-gap commits are all about dependency drift. The only thing that reliably brings the author back is the sytem breaking compatibility. No new contributors appear, so I assume this is some sort of hobby or academic maintenance project with no continuous development.",

    "stan-dev_cmdstanpy":
        "No qualifying inactivity gap (longest run of zero-commit months was 2), so this project fits the always active situation. Activity is irregular but sustained, with multiple maintainers and external contributors producing a steady mix of bug fixes, feature development, and documentation updates year over year. The absence of long gaps reflects how many people worked on teh project as when one person steps back, someone else is always committing. Recent activity continues at a high rate (most recent commit is within days of the assignment date).",

    "thelfer_mfrontgenericinterfacesupport":
        "A U-shaped pattern seems to fit this project as solid activity can be seen from 2018 to 2022, a 5-month gap in mid-to-late 2022, then a strong comeback. The gap was easy to interpret as pre-gap commits were formatting and cleanup work, while the first post-gap commits are a feature, which seems to suggest the maintainer came back specifically because a new feature was planned. Recovery involved both the same maintainer and external collaborators, indicating that the project is watched over by an active user community rather than solo effort.",

    "xeokit_xeokit-sdk":
        "No inactivity gap at all across the entire 2018 to 2026 history, so this also fits the 'active always' case. The activity pattern is steady and huge, with around 14856 commits and 116 authors in total. These authors produce continuous features, releases, and documentation updates driven by both the commercial maintainer and the user community. A project of this scale and funding doesn't seem to go dormant as there's always a new feature or release being worked on. There is nothing to interpret regarding gaps.",
}

for prj, note in notes.items():
    stats.loc[stats["Project"] == prj, "Notes"] = note

stats.to_csv("mp2_outputs/cdamron3_project_stats.csv", sep=";", index=False)
print("Notes updated for", len(notes), "projects.")

for _, r in stats.iterrows():
    print(f"\n=== {r['Project']} ===")
    print(r["Notes"])

Notes updated for 10 projects.

=== jhuapl-boss_intern ===
Activity is within bursts and declining relative to its early peak as numerous amounts of of feature and bug-fix commits are separated by long quiet stretches. The 5-month gap (2024-01 to 2024-05) was easy to interpret as the last pre-gap commits were a release tag followed by small fixes and CI cleanups. This makes sense as a natural wind-down after finishing a work cycle, and the very first post-gap commit shows the same two maintainers returning when a new bug surfaced. Recovery was fast and did not involve new contributors, suggesting the project is simply low-traffic between requests rather than abandoned.

=== jonescompneurolab_hnn ===
The project shows a clear two-phase pattern: high sustained activity (2017 to 2021) followed by a sharp drop-off and a 13-month silence. This gap was quite hard to interpret from commit messages alone as the last pre-gap commits were documentation edits and something about a WIP, which hint

In [17]:
os.makedirs("mp2_outputs/reflections", exist_ok=True)

for prj, note in notes.items():
    path = f"mp2_outputs/reflections/cdamron3_reflection_{prj}.md"
    with open(path, "w") as f:
        f.write(f"# Reflection: {prj}\n\n")
        f.write(note + "\n")
    print("Wrote", path)

Wrote mp2_outputs/reflections/cdamron3_reflection_jhuapl-boss_intern.md
Wrote mp2_outputs/reflections/cdamron3_reflection_jonescompneurolab_hnn.md
Wrote mp2_outputs/reflections/cdamron3_reflection_lpwgroup_crank.md
Wrote mp2_outputs/reflections/cdamron3_reflection_marianattestad_ribbon.md
Wrote mp2_outputs/reflections/cdamron3_reflection_mirnylab_cooler.md
Wrote mp2_outputs/reflections/cdamron3_reflection_privefl_bigstatsr.md
Wrote mp2_outputs/reflections/cdamron3_reflection_rveltz_piecewisedeterministicmarkovprocesses.jl.md
Wrote mp2_outputs/reflections/cdamron3_reflection_stan-dev_cmdstanpy.md
Wrote mp2_outputs/reflections/cdamron3_reflection_thelfer_mfrontgenericinterfacesupport.md
Wrote mp2_outputs/reflections/cdamron3_reflection_xeokit_xeokit-sdk.md


In [16]:
stats = pd.read_csv("mp2_outputs/cdamron3_project_stats.csv", sep=";")

# Force object dtype so ints can be written into a possibly-NaN column
stats["NumberOfGapsInTimeline"] = stats["NumberOfGapsInTimeline"].astype("object")

gap_counts = {
    "jhuapl-boss_intern": 6,
    "jonescompneurolab_hnn": 3,
    "lpwgroup_crank": 6,
    "marianattestad_ribbon": 7,
    "mirnylab_cooler": 5,
    "privefl_bigstatsr": 4,
    "rveltz_piecewisedeterministicmarkovprocesses.jl": 10,
    "stan-dev_cmdstanpy": 0,
    "thelfer_mfrontgenericinterfacesupport": 2,
    "xeokit_xeokit-sdk": 0,
}

for prj, n in gap_counts.items():
    stats.loc[stats["Project"] == prj, "NumberOfGapsInTimeline"] = n

stats.to_csv("mp2_outputs/cdamron3_project_stats.csv", sep=";", index=False)
print("Fixed NumberOfGapsInTimeline")
print(stats[["Project", "NumberOfGapsInTimeline"]])

Fixed NumberOfGapsInTimeline
                                           Project NumberOfGapsInTimeline
0                               jhuapl-boss_intern                      6
1                            jonescompneurolab_hnn                      3
2                                   lpwgroup_crank                      6
3                            marianattestad_ribbon                      7
4                                  mirnylab_cooler                      5
5                                privefl_bigstatsr                      4
6  rveltz_piecewisedeterministicmarkovprocesses.jl                     10
7                               stan-dev_cmdstanpy                      0
8            thelfer_mfrontgenericinterfacesupport                      2
9                                xeokit_xeokit-sdk                      0


In [20]:
stats = pd.read_csv("mp2_outputs/cdamron3_project_stats.csv", sep=";")

for col in ["LongestGapStart", "LongestGapEnd", "LongestGapLength"]:
    stats[col] = stats[col].astype("object")
    stats.loc[stats["Project"] == "stan-dev_cmdstanpy", col] = "N/A"

stats.to_csv("mp2_outputs/cdamron3_project_stats.csv", sep=";", index=False)
print(stats[stats["Project"] == "stan-dev_cmdstanpy"][
    ["Project", "LongestGapStart", "LongestGapEnd", "LongestGapLength"]
])

              Project LongestGapStart LongestGapEnd LongestGapLength
7  stan-dev_cmdstanpy             N/A           N/A              N/A
